In [0]:
%pip install dotenv
%pip install langchain-community
%pip install pymupdf
%pip install langchain-openai==0.3.21
%restart_python


In [0]:
import os, csv
from pathlib import Path
from typing import List
from dotenv import load_dotenv
from langchain.document_loaders import PyMuPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import random


In [0]:
def find_pdfs(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    # The rglob pattern is case-sensitive by default; to catch .PDF too, you could check suffix lowercased:
    pdfs = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf']
    return pdfs

load_dotenv()

try:
    os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
except NameError:
    raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")

In [0]:
t = []
files = find_pdfs('./data/')

for i in files:
    FILE_PATH = str(i)  

    doc = PyMuPDFLoader(FILE_PATH, mode='page') # open a document
    text = doc.load()

    if len(text) > 3:
        selected_pages = random.sample(text, 3)
    else:
        selected_pages = text

    for page in selected_pages:
        page.metadata['page'] += 1
    t.extend(selected_pages)

print(len(t))

leng = 0
for i in t:
    leng += len(i.page_content)
print(leng)

In [0]:
for doc in t:
    doc.metadata = {
        'page': doc.metadata.get('page'),
        'source': doc.metadata.get('source')
    }
t

In [0]:
llm = ChatOpenAI(model='gpt-4o-mini')


data = [
    ['Question', 'Document', 'Page', 'Answer']
]

for i in range(len(t)):
    doc_piece = t[i]

    question = PromptTemplate.from_template("Here is a page from a pdf document: {input}, write me a in dempth complicated question using the information from this page. The question must be a sentence long and must have any filler. This answer must really be just the question which is a sentence long.")

    chain1 = question | llm
    q = chain1.invoke(
        {
            "input": doc_piece
        }
    ).content

    answer = PromptTemplate.from_template("Answer this question: {input} using the information from this document: {doc}. Make your answer as short and precise as possible.")

    print(q)

    chain2 = answer | llm

    a = chain2.invoke(
        {
            "input": q,
            "doc": doc_piece
        }
    ).content

    data.append([q, doc_piece.metadata['source'], doc_piece.metadata['page'], a])

with open('questions.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)